The link for this information: https://neurosity.co/guides/machine-learning-eeg-classification-introduction
The Pipeline: From Skull to Label
Before we talk about specific algorithms, you need the mental model. Every EEG classification system, from a PhD student's MATLAB script to the Neurosity Crown's N3 chipset, follows the same basic pipeline. Understanding this pipeline is the trunk of the tree. Everything else is branches.

Step 1: Record the EEG. Electrodes on the scalp pick up voltage fluctuations. The Crown uses 8 channels at positions CP3, C3, F5, PO3, PO4, F6, C4, and CP4, sampling 256 times per second. Each sample is a vector of 8 numbers representing the voltage at each electrode. Over one second, that's 2,048 numbers.

Step 2: Preprocess. The raw signal is messy. You filter out frequencies you don't care about (typically bandpass to 1-50 Hz). You remove artifacts from eye blinks and muscle movements. You might re-reference the channels. Preprocessing doesn't add information, but it removes noise that would confuse everything downstream.

Step 3: Extract features. This is where the magic starts. You take a window of preprocessed EEG (say, 2 seconds of data) and compute numbers that describe what's happening in that window. Power in the alpha band. Ratio of theta to beta. Coherence between two channels. These numbers are your features, and they compress 4,096 raw data points into maybe 20-50 meaningful measurements.

Step 4: Classify. Feed those features into a machine learning algorithm that's been trained to map feature vectors to labels. "Focused." "Relaxed." "Left hand motor imagery." The algorithm outputs a prediction.

Step 5: Use the prediction. Trigger a notification, adjust music tempo, move a cursor on screen, or log data for later analysis.

That's it. Five steps. Each one matters enormously, and screwing up any single step can make the whole pipeline useless. But the two steps that determine whether your system actually works are 3 and 4. Feature extraction and classification. Let's go deep on both.



Feature Extraction: Teaching Your Model What to Look At
Raw EEG data is terrible input for a classifier. Not because it lacks information, but because it has too much, spread across too many dimensions, buried under too much noise. A 2-second window from 8 channels at 256Hz gives you 4,096 numbers. Most of those numbers are redundant. Many are noise. A few contain the signal you care about.

Feature extraction is the art of boiling those 4,096 numbers down to the 20-50 that matter.



The most common and most reliable EEG features are spectral. You decompose each channel's signal into frequency components (usually with a Fast Fourier Transform or Welch's method) and compute the power in standard frequency bands.

Band	Frequency Range	Associated States	Common Use in Classification
Delta	0.5-4 Hz	Deep sleep, unconsciousness	Sleep staging, anesthesia depth
Theta	4-8 Hz	Drowsiness, memory encoding, meditation	Attention monitoring, meditation detection
Alpha	8-13 Hz	Relaxed wakefulness, eyes closed, inhibition	Relaxation detection, workload estimation
Beta	13-30 Hz	Active thinking, focus, motor planning	Focus detection, motor imagery
Gamma	30-50 Hz	Cross-modal integration, higher cognition	Cognitive load, binding/perception tasks


The Classifiers: Four Algorithms That Actually Work
There are dozens of ML algorithms you could throw at EEG features. But in practice, four dominate the literature and for good reason. Each has a specific strength that maps to a specific EEG challenge.

*LDA is the one that Mark explained to us* Linear Discriminant Analysis (LDA)
LDA is the workhorse of BCI classification, and it has been since the 1990s. The idea is elegant: find the linear combination of features that best separates two classes. If you're classifying "focused" vs "relaxed," LDA finds the axis in feature space where the two states are maximally spread apart and minimally spread within each group.

Why does LDA dominate BCI? Three reasons. First, it's fast. On embedded hardware, LDA classification takes microseconds. Second, it has very few parameters to tune, which means it's hard to overfit (more on that later). Third, it works surprisingly well with small datasets because it makes a strong assumption (equal covariance matrices for each class) that acts as a built-in regularizer.

The downside? LDA can only draw straight lines between classes. If the true boundary between "focused" and "relaxed" in feature space is curved or wiggly, LDA will miss it. For many EEG tasks, the linear assumption holds well enough. For complex multi-class problems, it starts to struggle.

Support Vector Machines (SVM)
SVMs are what you reach for when LDA's linear boundary isn't enough. An SVM finds the hyperplane that separates two classes with the maximum margin, the widest possible gap between the nearest data points of each class. But the real power comes from the kernel trick: by projecting your features into a higher-dimensional space (without actually computing the projection, which is the trick part), SVMs can learn nonlinear decision boundaries.

For EEG, the radial basis function (RBF) kernel is the go-to. It lets the SVM carve out curved, flexible boundaries in feature space. The cost is two hyperparameters (C and gamma) that need tuning, which means you need proper cross-validation (seriously, we'll get there).

SVMs with RBF kernels consistently rank among the top classifiers in BCI competitions. They're particularly strong for motor imagery classification, where the boundaries between "imagine moving left hand" and "imagine moving right hand" in feature space aren't cleanly linear.

Random Forest
A Random Forest is an ensemble of decision trees, typically hundreds of them, each trained on a random subset of your data and a random subset of your features. To classify a new sample, every tree votes, and the majority wins.

Random Forests have a property that makes them especially attractive for EEG work: they're resistant to noisy features. If 30 of your 50 features are garbage, a Random Forest will naturally figure this out and lean on the 20 that matter. The trees that happen to use informative features will agree with each other. The trees that use noise will disagree. The consensus filters out the junk.

They also give you feature importance scores for free. After training, you can ask "which features contributed most to classification?" This is invaluable for EEG, where knowing that theta/beta ratio matters more than gamma power for your specific task helps you understand the neuroscience, not just the accuracy.

k-Nearest Neighbors (kNN)
kNN is the simplest classifier here and the most underrated for certain EEG tasks. To classify a new sample, kNN finds the k most similar samples in the training set and assigns the majority label. That's it. No training phase, no model parameters, no assumptions about the data distribution.

kNN works well for EEG when you have a relatively small, clean feature set and you're doing subject-dependent classification (more on this distinction shortly). It's also useful as a baseline. If a more complex algorithm can't beat kNN on your dataset, something is probably wrong with your features, not your classifier.

The weakness? kNN scales poorly. It has to store and search the entire training set at prediction time. For real-time BCI with thousands of training samples, this can be too slow. And it's extremely sensitive to irrelevant features, since distances are computed across all dimensions, noisy features can dominate the neighbor calculation.

